# Categorical Encoding Techniques

Demonstrates various categorical encoding techniques for machine learning. Categorical variables (like "Color", "Department", "Make") need to be converted to numerical form for ML models.

## Key Techniques Covered

1. **One-Hot Encoding** - Binary columns for each category (low cardinality)
2. **Hash Encoding** - Fixed-size buckets via hashing (high cardinality, streaming)
3. **Target Encoding** - Replace with mean of target variable (high signal features)
4. **Label/Ordinal Encoding** - Integer assignment (tree models, ranked data)
5. **Binary Encoding** - Binary representation (memory-balanced)
6. **Count/Frequency Encoding** - Popularity signal
7. **Leave-One-Out Encoding** - Target encoding variant (reduces leakage)

## Dataset

Uses EPA Fuel Economy data with categorical features: `make` (manufacturer), `model` (vehicle model), `VClass` (vehicle class), `fuelType` (fuel type), `trans_dscr` (transmission description).

## Key Considerations

- **Cardinality matters**: Low cardinality (<10 unique values) → One-Hot. High cardinality (>50) → Hash or Target
- **Target encoding risks**: Can cause data leakage if not done carefully (use cross-validation)
- **Tree models**: Can handle Label encoding better than linear models
- **Production**: Hash encoding is memory-efficient for streaming data

| Encoder | Logic | Best Use Case |
| :--- | :--- | :--- |
| **One-Hot** | Creates a new binary column (0 or 1) for every unique category in the feature. | **Low Cardinality:** Ideal for features with few unique values (e.g., "Color" or "Department"). |
| **Hash** | Uses a mathematical function to map categories into a fixed number of "buckets" or columns. | **High Cardinality/Streaming:** Best for massive datasets where you must cap memory usage. |
| **Target** | Replaces a category with the (blended) mean of the target variable for that group. | **High Cardinality/High Signal:** Best when the category name itself directly correlates to the outcome. |

---

### Other Encoders

*   **Label / Ordinal:** Assigns a unique integer to each category; best for tree-based models or data with a natural rank (e.g., "Bronze", "Silver", "Gold").
*   **Binary:** Converts categories into binary code and splits the digits into a small number of columns to balance memory and information.
*   **Count / Frequency:** Replaces the category name with its total number of occurrences to capture "popularity" as a numerical signal.
*   **Leave-One-Out:** A variation of Target Encoding that calculates the mean excluding the current row's value to further reduce data leakage.

In [ ]:
import sys, pathlib
_vsc_nb = globals().get('__vsc_ipynb_file__')
_nb_dir = pathlib.Path(_vsc_nb).parent if _vsc_nb else pathlib.Path.cwd()
if str(_nb_dir) not in sys.path:
    sys.path.insert(0, str(_nb_dir))

import pandas as pd
import numpy as np
from utils import load_fuel_economy_data

In [ ]:
# analyzing the data
df = load_fuel_economy_data()

df.head()

In [ ]:
df.VClass

In [ ]:
# create a column per unique value of df.vClass
pd.get_dummies(df.VClass)

In [ ]:
# pick object/str unique types, doesn't make sense for numeric/continuous values
(
    df.select_dtypes(object).nunique().index
)

In [ ]:
df.info()

In [ ]:
# Deploying one-hot encoding
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.pipeline import FunctionTransformer, Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder

X=df.drop(columns=['city08', 'highway08', 'comb08', 'createdOn'])
y=df.city08

X_train, X_test, y_train, y_test = train_test_split(X, np.log(y), random_state=42)

cylinders_imputer = SimpleImputer(strategy='constant', fill_value=0)
disply_imputer = SimpleImputer(strategy='median')
std_scaler = StandardScaler()
cat_imputer = SimpleImputer(strategy='constant', fill_value='missing')
one_hot_encoder = OneHotEncoder(drop='first', max_categories=10, sparse_output=False, handle_unknown='ignore') # pick only 10 cats, ignore the rest


cat_cols = ['make', 'model', 'trany', 'drive',
        'VClass', 'eng_dscr', 'evMotor', 'fuelType', 'trans_dscr']

# Encoding needs to happen in sequence (after or before the transformer steps)
# Why? because everything within a step is executed parallely (cyl_imputer and displ_imputer)
cat_pipeline = Pipeline([
    ('cat_imputer', cat_imputer),
    ('one_hot_encoder', one_hot_encoder)
])

preprocessor = ColumnTransformer(
    transformers = [
        ('cyl_imputer', cylinders_imputer, ['cylinders']),
        ('displ_imputer', disply_imputer, ['displ']),
        ('cat_pipeline', cat_pipeline, cat_cols)
    ],
    remainder='passthrough'
)

# take the X and set it to whatever variable name 'name' you pass to it
def debug_transformer(X, name):
    globals()[name]=X
    return X

# train the model
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    # if we print tmp_X now, function transformer will have stored the intermediate value of X between preprocessor and std_scaler in it
    ('debug', FunctionTransformer(debug_transformer, kw_args={'name':'tmp_X'})),
    ('std_scaler', std_scaler),
    ('lr', LinearRegression())])


pipeline.fit(X_train, y_train)


# evaluate the model

print(f"R2 score is {pipeline.score(X_test, y_test)}")
print(f"mse is {mean_absolute_error(y_test, pipeline.predict(X_test))}")


In [ ]:
!pip install category_encoders

In [ ]:
# Hash Encoding
high_cardinality_cols = (
    df.select_dtypes(object)
    .nunique()
    .pipe(lambda s: s[s>40]) # only pick high cardinality values i.e., columns with >40 unique values
    .index
).to_numpy()

low_cardinality_cols =  (
    df.select_dtypes(object)
    .nunique()
    .pipe(lambda s: s[s<40]) # only pick high cardinality values i.e., columns with >40 unique values
    .index
).to_numpy()


high_cardinality_cols=list(high_cardinality_cols)
low_cardinality_cols=list(low_cardinality_cols)

high_cardinality_cols



In [ ]:
# Exploring hashencoding
from category_encoders import hashing
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.pipeline import FunctionTransformer, Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder

X=df.drop(columns=['city08', 'highway08', 'comb08', 'createdOn'])
y=df.city08

X_train, X_test, y_train, y_test = train_test_split(X, np.log(y), random_state=42)

cylinders_imputer = SimpleImputer(strategy='constant', fill_value=0)
disply_imputer = SimpleImputer(strategy='median')
std_scaler = StandardScaler()
one_hot_encoder = OneHotEncoder(drop='first', max_categories=10, sparse_output=False, handle_unknown='ignore') # pick only 10 cats, ignore the rest
hashing_encoder = hashing.HashingEncoder(n_components=10, drop_invariant=True) # for high cardinality values use a hash function to map the entire cardinality to a fixed set of cols (10 here)

preprocessor = ColumnTransformer(
    transformers = [
        ('cyl_imputer', cylinders_imputer, ['cylinders']),
        ('displ_imputer', disply_imputer, ['displ']),
        ('one_hot_encoder', one_hot_encoder, low_cardinality_cols),
        ('hashing_encoder', hashing_encoder, high_cardinality_cols),
    ],
    remainder='drop'
)

# take the X and set it to whatever variable name 'name' you pass to it
def debug_transformer(X, name):
    globals()[name]=X
    return X

# train the model
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    # if we print tmp_X now, function transformer will have stored the intermediate value of X between preprocessor and std_scaler in it
    ('debug', FunctionTransformer(debug_transformer, kw_args={'name':'tmp_X'})),
    ('std_scaler', std_scaler),
    ('lr', LinearRegression())])


pipeline.fit(X_train, y_train)


# evaluate the model

print(f"R2 score is {pipeline.score(X_test, y_test)}")
print(f"mse is {mean_absolute_error(y_test, pipeline.predict(X_test))}")


Target Encoding transforms categorical labels into a statistical "track record." It maps features into high-signal "buckets" representing the likelihood of producing specific target ranges, effectively pre-calculating the relationship so the model doesn't have to.

To prevent "perfect hindsight," you blend the specific categorical mean with the global target mean (the baseline for all data). This acts as a stabilizer: it pulls noisy, low-volume categories toward the global norm, forcing the model to learn broader patterns rather than simply memorizing flukes.

In [ ]:
# Target Encoding
from sklearn.preprocessing import TargetEncoder

tc = TargetEncoder(target_type='continuous', random_state=42)
tc.fit_transform(X_train[['make']], y_train)

In [ ]:
# Exploring target encoding


from category_encoders import hashing
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.pipeline import FunctionTransformer, Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder

X=df.drop(columns=['city08', 'highway08', 'comb08', 'createdOn'])
y=df.city08

X_train, X_test, y_train, y_test = train_test_split(X, np.log(y), random_state=42)

cylinders_imputer = SimpleImputer(strategy='constant', fill_value=0)
disply_imputer = SimpleImputer(strategy='median')
std_scaler = StandardScaler()
one_hot_encoder = OneHotEncoder(drop='first', max_categories=10, sparse_output=False, handle_unknown='ignore') # pick only 10 cats, ignore the rest
hashing_encoder = hashing.HashingEncoder(n_components=10, drop_invariant=True) # for high cardinality values use a hash function to map the entire cardinality to a fixed set of cols (10 here)


preprocessor = ColumnTransformer(
    transformers = [
        ('cyl_imputer', cylinders_imputer, ['cylinders']),
        ('displ_imputer', disply_imputer, ['displ']),
        ('one_hot_encoder', one_hot_encoder, low_cardinality_cols),
        #('hashing_encoder', hashing_encoder, high_cardinality_cols),
        ('target_encoder', tc, high_cardinality_cols),
    ],
    remainder='drop'
)

# take the X and set it to whatever variable name 'name' you pass to it
def debug_transformer(X, name):
    globals()[name]=X
    return X

# train the model
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    # if we print tmp_X now, function transformer will have stored the intermediate value of X between preprocessor and std_scaler in it
    ('debug', FunctionTransformer(debug_transformer, kw_args={'name':'tmp_X'})),
    ('std_scaler', std_scaler),
    ('lr', LinearRegression())])


pipeline.fit(X_train, y_train)


# evaluate the model

print(f"R2 score is {pipeline.score(X_test, y_test)}")
print(f"mse is {mean_absolute_error(y_test, pipeline.predict(X_test))}")


In [ ]:
cat_cols = ['trany', 'drive', 'VClass', 'eng_dscr', 'evMotor', 'fuelType', 'trans_dscr']
high_cardinality_cols=[]
low_cardinality_cols=[]
for col in cat_cols:
    if df[col].nunique() < 40:
        low_cardinality_cols.append(col)
    else:
        high_cardinality_cols.append(col)

low_cardinality_cols

In [ ]:
# Exercise

from category_encoders import hashing
from sklearn.compose import ColumnTransformer
from sklearn import set_config
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder

set_config(transform_output='pandas')

X=df.drop(columns=['city08', 'highway08', 'comb08', 'createdOn'])
y=df.city08

X_train, X_test, y_train, y_test = train_test_split(X, np.log(y), random_state=42)

std_scaler = StandardScaler()
one_hot_encoder = OneHotEncoder(drop='first', max_categories=10, sparse_output=False, handle_unknown='ignore') # pick only 10 cats, ignore the rest
hashing_encoder = hashing.HashingEncoder(n_components=10, drop_invariant=True) # for high cardinality values use a hash function to map the entire cardinality to a fixed set of cols (10 here)
target_encoder = TargetEncoder(target_type='continuous', random_state=42)

preprocessor = ColumnTransformer(
    transformers = [
        ('one_hot_encoder', one_hot_encoder, low_cardinality_cols),
        #('hashing_encoder', hashing_encoder, high_cardinality_cols),
        ('target_encoder', target_encoder, high_cardinality_cols),
    ],
    remainder='drop'
)

# train the model
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('std_scaler', std_scaler),
    ('lr', LinearRegression())])


pipeline.fit(X_train, y_train)

# evaluate the model

print(f"R2 score is {pipeline.score(X_test, y_test)}")
print(f"mse is {mean_absolute_error(y_test, pipeline.predict(X_test))}")


There are specific column level transformations we may need to apply
For eg. if a car is an EV indicated by evMotor and eng_dscr is empty, we may want to add 'EV', if it is not an EV (indicated by some other value in the evMotor column)we may need to add 'FFS'.
Such business-logic dependent transformations require custom transformers, this eliminates the need for a separate data-cleaning step external to the pipeline.

Below is an AI generated snippet to do so

There are specific column level transformations we may need to apply
For eg. if a car is an EV indicated by evMotor and eng_dscr is empty, we may want to add 'EV', if it is not an EV (indicated by some other value in the evMotor column)we may need to add 'FFS'.
Such business-logic dependent transformations require custom transformers, this eliminates the need for a separate data-cleaning step external to the pipeline.

Below is an AI generated snippet to do so

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin

class EngineDescImputer(BaseEstimator, TransformerMixin):
    def __init__(self):
        pass

    def fit(self, X, y=None):
        # Nothing to learn from the data, just return self
        return self

    def transform(self, X):
        # Create a copy to avoid SettingWithCopy warnings
        X = X.copy()

        # Logic: If eng_dscr is missing:
        # Check evMotor. If NaN -> NOTAVL, else -> EV
        mask = X['eng_dscr'].isna()

        X.loc[mask, 'eng_dscr'] = X.loc[mask, 'evMotor'].apply(
            lambda x: 'EV' if pd.notna(x) else 'NOTAVL'
        )

        # Return only the column we need for the next step in the pipeline
        # or the whole dataframe depending on your ColumnTransformer setup
        return X[['eng_dscr']]

**Feature extraction**

Feature extraction is the process of transforming raw data into features that better represent the underlying problem to the predictive models, resulting in improved model accuracy on the unseen data


**PCA**

*Principal Component Analysis (PCA)* is a technique for reducing the dimensionality of data. It can also remove noise and might be useful as feature engineering for oither algorithms.

In [ ]:
# Exercise

from category_encoders import hashing
from sklearn.compose import ColumnTransformer
from sklearn import set_config
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.pipeline import FunctionTransformer, Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.preprocessing import TargetEncoder
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer

set_config(transform_output='pandas')

X=df.drop(columns=['city08', 'highway08', 'comb08', 'createdOn'])
y=df.city08

X_train, X_test, y_train, y_test = train_test_split(X, np.log(y), random_state=42)

std_scaler = StandardScaler()
cylinders_imputer = SimpleImputer(strategy='constant', fill_value=0)
disply_imputer = SimpleImputer(strategy='median')
one_hot_encoder = OneHotEncoder(drop='first', max_categories=10, sparse_output=False, handle_unknown='ignore') # pick only 10 cats, ignore the rest
hashing_encoder = hashing.HashingEncoder(n_components=10, drop_invariant=True) # for high cardinality values use a hash function to map the entire cardinality to a fixed set of cols (10 here)
target_encoder = TargetEncoder(target_type='continuous', random_state=42)

preprocessor = ColumnTransformer(
    transformers = [
        ('cyl_imputer', cylinders_imputer, ['cylinders']),
        ('displ_imputer', disply_imputer, ['displ']),
        ('one_hot_encoder', one_hot_encoder, low_cardinality_cols),
        #('hashing_encoder', hashing_encoder, high_cardinality_cols),
        ('target_encoder', target_encoder, high_cardinality_cols),
    ],
    remainder='drop'
)

# take the X and set it to whatever variable name 'name' you pass to it
def debug_transformer(X, name):
    globals()[name]=X
    return X

# train the model
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('std_scaler', std_scaler),
    ('pca', PCA(n_components=10)),
    ('debug', FunctionTransformer(debug_transformer, kw_args={'name':'tmp_X'})),
    ('lr', LinearRegression())])


pipeline.fit(X_train, y_train)

# evaluate the model

print(f"R2 score is {pipeline.score(X_test, y_test)}")
print(f"mse is {mean_absolute_error(y_test, pipeline.predict(X_test))}")


In [ ]:
tmp_X

# This gives the Principal components, but not their labels or any intuitive knowledge about it